# Analyse de sensibilité — Étape 4

Que devient l'avantage de chaque stratégie quand les paramètres du système changent ?
L'Étape 3 a établi que le solveur reproduit le papier ; cette étape s'en sert pour
explorer, et pour en tirer des recommandations exploitables.

Sauf mention contraire, les paramètres non balayés gardent leur valeur de référence
(§5.3) : `Λ̄ = 1`, `a = 1`, `c = 0`, `h = 0.01`, `q = 0.01`, `µ = 0.11`, `ε = 0.8`.
Le point de fonctionnement `ε = 0.8` est celui où les stratégies se séparent le plus,
donc celui où les écarts sont les plus lisibles.

In [ ]:
import json, sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import sensitivity as S

if S.RESULTS_PATH.exists():
    results = json.loads(S.RESULTS_PATH.read_text(encoding="utf-8"))
    print(f"Résultats chargés depuis {S.RESULTS_PATH.name}")
else:
    results = S.run_all()
    print("Résultats recalculés")

sweeps = {k: pd.DataFrame(v) for k, v in results["sweeps"].items()}
pd.set_option("display.width", 140)

## 1. Un seul environnement : le plafond de la tarification dynamique

Sans fluctuation de l'environnement, S, SB, SP et EDP se confondent — un seul prix, un
seul base-stock. Le profit gain de DP mesure donc exactement ce que rapporte le fait de
**faire dépendre le prix du niveau de stock**, et rien d'autre.

In [ ]:
f1 = pd.DataFrame(results["figure_1"])
fig, ax = plt.subplots(figsize=(8, 4.5))
for h, grp in f1.groupby("h"):
    ax.plot(grp["mu"], grp["pg_dp_s"], marker="o", ms=3, label=f"h = {h}")
ax.set_xlabel("taux de service $\\mu$")
ax.set_ylabel("$PG^{dp,s}$ (%)")
ax.set_title("Figure 1 — apport de la tarification dynamique, un seul environnement")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

peak = f1.loc[f1["pg_dp_s"].idxmax()]
print(f"gain maximal observé : {peak['pg_dp_s']:.2f} % "
      f"(µ={peak['mu']}, h={peak['h']}) — papier §5.2.1 : 3.81 %")

**Ce que ça dit.** Le gain est faible et non monotone en `µ` : il s'annule aux deux
extrémités et culmine à capacité intermédiaire. Aux deux bouts, la raison est différente
mais le résultat identique — à capacité très faible le stock reste quasi nul et l'on vend
en permanence au prix fort, à capacité abondante le réapprovisionnement suffit à lui seul
à absorber l'aléa. La tarification n'a de valeur que dans l'entre-deux, là où le stock
fluctue assez pour porter de l'information.

Le papier établit que le maximum atteignable sur *tout* le plan `(µ, h)` est de **3.81 %**
(§5.2.1). C'est un plafond, et il est bas.

> **Enseignement.** Dans un système à demande stationnaire, faire varier le prix selon le
> stock ne rapporte au mieux que quelques pourcents. Si la demande ne fluctue pas, le jeu
> n'en vaut pas la chandelle : un prix statique bien choisi suffit.

## 2. Deux environnements : le balayage du taux de service

In [ ]:
mu = sweeps["mu"]
fig, ax = plt.subplots(figsize=(8, 4.5))
for col, label in [("pg_dp_s", "DP"), ("pg_edp_s", "EDP"),
                   ("pg_sb_s", "SB"), ("pg_sp_s", "SP")]:
    ax.plot(mu["value"], mu[col], marker="o", ms=3, label=label)
ax.set_xlabel("taux de service $\\mu$")
ax.set_ylabel("profit gain sur S (%)")
ax.set_title("Balayage de $\\mu$  ($\\varepsilon = 0.8$)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()
display(mu[["value", "pg_sp_s", "pg_sb_s", "pg_edp_s", "pg_dp_s", "pg_dp_edp"]])

**Ce que ça dit.** Les ordres de grandeur changent complètement dès que la demande
fluctue : là où la tarification dynamique plafonnait à 3.81 % en mono-environnement, elle
dépasse ici 13 %. Mais l'essentiel de ce gain n'est **pas** dû à la dépendance au stock —
la colonne `pg_dp_edp` reste sous 3.3 % sur tout le balayage.

Autrement dit : la valeur vient de l'**adaptation à l'environnement**, pas de l'adaptation
au niveau de stock. Le premier axe rapporte gros, le second marginalement.

Noter aussi que SP — prix unique, base-stock par environnement — reste très proche de S.
Ajuster ses stocks sans ajuster ses prix ne sert presque à rien.

## 3. Coût de stockage, variabilité, vitesse des transitions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, xlabel) in zip(axes, [("h", "coût de stockage $h$"),
                                     ("eps", "variabilité $\\varepsilon$"),
                                     ("q", "taux de transition $q$")]):
    df = sweeps[name]
    for col, label in [("pg_dp_s", "DP"), ("pg_edp_s", "EDP"), ("pg_sp_s", "SP")]:
        ax.plot(df["value"], df[col], marker="o", ms=4, label=label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("profit gain sur S (%)")
    if name in ("h", "q"):
        ax.set_xscale("log")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()

**Coût de stockage `h`.** Plus le stock coûte cher, plus la tarification vaut : elle
permet d'écouler avant que la facture de stockage ne tombe. À `h` très faible, on peut
stocker sans compter et le prix perd son rôle d'outil d'écoulement.

**Variabilité `ε`.** C'est le déterminant principal, et le plus net : à `ε = 0` tout gain
disparaît, et il croît continûment ensuite. Le gain de la tarification est, en substance,
une prime à l'incertitude de la demande.

**Vitesse des transitions `q`.** Le gain **décroît** quand l'environnement change vite
(§5.3.1). Deux régimes se lisent aux extrémités : quand `q` est faible, le système voit
deux sous-systèmes quasi indépendants et peut s'ajuster pleinement à chacun ; quand `q`
est grand, aucune tarification n'a le temps de produire ses effets avant que le contexte
ne bascule.

> **Enseignement.** La tarification dépendante de l'environnement se justifie quand les
> régimes de demande sont **marqués et durables**. Des saisons franches et longues, oui ;
> un bruit rapide, non — l'effort de pilotage ne sera pas récupéré.

## 4. Combien de prix faut-il vraiment ? (Table 2)

Question opérationnelle directe : changer de prix a un coût — étiquetage, systèmes,
réaction des clients. Si l'on se limite à deux ou trois tarifs au lieu d'un continuum,
que perd-on ?

Le cas à trois prix reprend la contrainte de l'Appendix E, `p3 = (p1+p2)/2`, qui en fait
une **borne inférieure** de l'optimum à trois prix — c'est cette borne qui figure dans la
Table 2 publiée, et la reproduire suppose d'appliquer la même contrainte.

In [ ]:
menu = pd.DataFrame(results["menu"])
display(menu)
print("part du bénéfice du continuum captée :")
print("  calculé :", {k: f"{v:.1%}" for k, v in results["menu_share"].items()})
print("  papier  :", {k: f"{v:.1%}" for k, v in results["published_menu_share"].items()})

> **Enseignement, et c'est le plus actionnable de l'étude.** Deux prix captent environ
> 78 % du bénéfice d'un continuum, trois en captent environ 92 %. Le rendement décroît
> très vite : l'essentiel se joue entre « un seul prix » et « deux prix ».
>
> Une grille à deux ou trois tarifs, lisible pour le client et simple à administrer, est
> donc préférable à un moteur de prix continu dans la quasi-totalité des cas.

## Synthèse — que recommander, et quand

| condition observée | stratégie recommandée | pourquoi |
|---|---|---|
| demande stable (`ε` faible) | **S** — prix unique | le gain de toute alternative est sous ~2 % |
| régimes marqués et durables (`ε` élevé, `q` faible) | **EDP** — un prix par régime | capte l'essentiel du gain, avec des changements de prix rares |
| capacité très contrainte ou très abondante | **S** | le gain retombe aux deux extrémités de `µ` |
| capacité intermédiaire, stockage coûteux | **EDP**, éventuellement DP | c'est le seul domaine où la dépendance au stock ajoute quelque chose |
| contrainte opérationnelle sur les changements de prix | **menu à 2 ou 3 prix** | ~78 % et ~92 % du bénéfice du continuum |

Le fil conducteur est constant sur les quatre balayages : **DP n'apporte que marginalement
plus qu'EDP**, tout en exigeant un pilotage en temps réel du stock, des changements de prix
fréquents et l'exposition à la réaction des clients. Le papier conclut de même (§5.3.4) —
EDP est le bon compromis.

La limite à garder en tête : le modèle ne facture ni le coût de mise en œuvre d'une
politique, ni la réaction des clients aux variations de prix. Les gains affichés pour DP
sont donc des **majorants**, et l'écart réel avec EDP est plus étroit encore que ce que
montrent ces courbes.